## Create a new environment in bioconda (miniconda)

In [4]:
!conda create -n 16s-nanopore -c bioconda -c conda-forge nanoplot cutadapt chopper kma emu osfclient -y
!conda activate 16s-nanopore

2 channel Terms of Service accepted
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.7.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/marcos/miniconda3/envs/16s-nanopore

  added / updated specs:
    - chopper
    - cutadapt
    - emu
    - kma
    - nanoplot
    - osfclient


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    brotli-python-1.2.0        |  py313h2fc2bef_3         359 KB  conda-forge
    charset-normalizer-3.5.0   |     pyhd8ed1ab_0          62 KB  conda-forge
    clang-22.1.8               |default_cfg_hecfc6a8_6          33 KB  conda-forge
    clang-22-22.1.8            |default_hd70ba2e_6         916 KB  conda-forge
    clang_impl_linux-64

In [5]:
# Sanity check using 'conda run' to point to the correct environment
!conda run -n 16s-nanopore NanoPlot --version
!conda run -n 16s-nanopore cutadapt --version
!conda run -n 16s-nanopore chopper --version
!conda run -n 16s-nanopore emu --version
!conda run -n 16s-nanopore osf -V
!conda run -n 16s-nanopore kma -v

NanoPlot 1.47.1
5.2
chopper 0.13.0
emu v3.6.2
usage: osf [-h] [-u USERNAME] [-p PROJECT] [-v]
           {clone,init,fetch,geturl,list,ls,upload,remove,rm} ...
osf: error: unrecognized arguments: -V
ERROR conda.cli.main_run:execute(142): `conda run osf -V` failed. (See above for error)
KMA-1.6.13


## Renamed

In [ ]:
import os
import shutil
import pandas as pd

# 1. Paths
folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s'
csv_file = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_16s.csv'
renamed_folder = os.path.join(folder, 'renamed')  # raw files stay untouched
os.makedirs(renamed_folder, exist_ok=True)

# 2. Read metadata (comma is the expected separator for a .csv)
df = pd.read_csv(csv_file, sep=',')
assert {'Name', 'sample_title'}.issubset(df.columns), f"Missing columns, found: {df.columns.tolist()}"

dupes = df['sample_title'][df['sample_title'].duplicated()].tolist()
if dupes:
    raise ValueError(f"Duplicate sample_title values would overwrite files: {dupes}")

log = []

# 3. Copy files under the new name, preserving the original extension
for _, row in df.iterrows():
    old_name = str(row['Name']).strip()
    sample_title = str(row['sample_title']).strip()
    old_path = os.path.join(folder, old_name)

    if not os.path.exists(old_path):
        print(f"[ERROR] File not found: {old_name}")
        continue

    if old_name.endswith('.fastq.gz'):
        ext = '.fastq.gz'
    elif old_name.endswith('.fastq'):
        ext = '.fastq'
    else:
        ext = os.path.splitext(old_name)[1]

    new_name = f"{sample_title}{ext}"
    new_path = os.path.join(renamed_folder, new_name)

    shutil.copy2(old_path, new_path)
    log.append({'old_name': old_name, 'new_name': new_name})
    print(f"[SUCCESS] Copied: {old_name} -> {new_name}")

pd.DataFrame(log).to_csv(os.path.join(renamed_folder, 'rename_manifest.csv'), index=False)
print("Done!")

## Library and directory preparation

In [ ]:
import os
import subprocess

# 1. Paths configuration (Updating based on your previous paths)
renamed_folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed'
base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream'

# Define specific output folders for each step
qc_dir = os.path.join(base_out, '1_qc_raw')
trimmed_dir = os.path.join(base_out, '2_trimmed')
filtered_dir = os.path.join(base_out, '3_filtered')
emu_dir = os.path.join(base_out, '4_emu_taxa')

# Create directories if they don't exist
for directory in [qc_dir, trimmed_dir, filtered_dir, emu_dir]:
    os.makedirs(directory, exist_ok=True)

# 2. Get list of renamed files
fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]
print(f"Found {len(fastq_files)} fastq files to process.")

## Quality control of the reads

In [ ]:
from pathlib import Path
import subprocess

env = "16s-nanopore"
threads = "4"

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed")
qc_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/1_qc_raw")
qc_dir.mkdir(parents=True, exist_ok=True)

# Removido o multiqc da instalação
subprocess.run(
    f"conda install -n {env} -c bioconda -c conda-forge nanocomp -y",
    shell=True,
    check=True
)

fastq_files = sorted(
    file for file in renamed_folder.iterdir()
    if file.name.endswith((".fastq", ".fastq.gz"))
)

print(f"Starting QC for {len(fastq_files)} samples...")

for file in fastq_files:
    sample = file.name.split(".fastq")[0]
    print(f"Running NanoPlot (with dot plots): {sample}")

    subprocess.run([
        "conda", "run", "-n", env, "NanoPlot",
        "-t", threads, "--fastq", str(file),
        "-o", str(qc_dir / sample),
        "--plots", "dot"
    ], check=True)

nanocomp_out = qc_dir / "NanoComp_Report"
nanocomp_out.mkdir(exist_ok=True)

print("Generating NanoComp report...")
subprocess.run([
    "conda", "run", "-n", env, "NanoComp", "-t", threads,
    "--fastq", *map(str, fastq_files),
    "--names", *(file.name.split(".fastq")[0] for file in fastq_files),
    "-o", str(nanocomp_out)
], check=True)

print("\nQuality control completed!")
print(f"NanoComp report: {nanocomp_out}")

For decision trimmer, use the LengthvsQualityScatterPlot_kde, which is a tool that generates a scatter plot of read lengths versus quality scores. This can help identify any issues with the sequencing data, such as low-quality reads or unexpected length distributions.

## Remove primers

Use the next code to detect the primers in the reads and remove them. The command will generate a summary of the primer removal process, including the number of reads that were trimmed and the number of reads that were discarded due to low quality.

In [ ]:
import os
import gzip

# Pathway to the renamed folder containing FASTQ files
renamed_folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed'

# Primers to check
primers_to_check = {
    "27F (Standard)": "AGRGTTYGATYMTGGCTCAG",
    "515F (V4)": "GTGYCAGCMGCCGCGGTAA",
    "1492R (Standard)": "RGYTACCTTGTTACGACTT"
}

sample_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]
test_file = os.path.join(renamed_folder, sample_files[0])

print(f"Processing sample: {sample_files[0]}\n")

# Read the first 1000 sequences from the FASTQ file
sequences = []
open_func = gzip.open if test_file.endswith('.gz') else open
with open_func(test_file, 'rt') as f:
    for i, line in enumerate(f):
        if i % 4 == 1:  # Line of the DNA sequence in the FASTQ file
            sequences.append(line.strip())
        if len(sequences) >= 1000:
            break

# Count how many times the primer fragments appear at the beginning of the reads
for name, primer in primers_to_check.items():
    # Get the first 15 nucleotides of the primer for partial search
    primer_snippet = primer[:15].replace('R', 'A').replace('Y', 'C').replace('M', 'A').replace('T', 'T')
    matches = sum(1 for seq in sequences if primer_snippet in seq[:30])
    print(f"Primer {name}: Found in {matches} of 1000 tested reads.")

Remove primers from the reads using the following command:

In [ ]:
import os
import subprocess

subprocess.run("conda install -n 16s-nanopore -c bioconda cutadapt -y", shell=True, check=True, stdout=subprocess.DEVNULL)

renamed_folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed'
trimmed_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/2_trimmed'

os.makedirs(trimmed_dir, exist_ok=True)

fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]

# 4. Run Cutadapt to remove 16S primers
# Replace these sequences with the exact primers used by the sequencing company
primer_f = "AGRGTTYGATYMTGGCTCAG" # Example: 27F
primer_r = "RGYTACCTTGTTACGACTT"  # Example: 1492R

print("Starting Primer Trimming with cutadapt...")

for file in fastq_files:
    sample_name = file.split('.fastq')[0]
    input_path = os.path.join(renamed_folder, file)
    output_path = os.path.join(trimmed_dir, file) # keeping original extension
    
    cmd = [
        "conda", "run", "-n", "16s-nanopore",
        "cutadapt",
        "-g", primer_f,      # Forward primer
        "-a", primer_r,      # Reverse primer
        "-e", "0.15",        # Allowed error rate
        # "--discard-untrimmed", # Optional: throw away reads that don't have primers
        "-o", output_path,
        input_path
    ]
    
    print(f"Trimming {sample_name}...")
    subprocess.run(cmd, check=True)

print("Trimming finished!")

## Filter reads by length with chopper

Trim the reads to a specific length range using chopper. This step is important to ensure that only reads of the desired length are retained for downstream analysis. The command will filter the reads based on the specified minimum and maximum length thresholds.

In [ ]:
# 5. Run Chopper for quality and length filtering
import os
import subprocess

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed")
filtered_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")
filtered_dir.mkdir(parents=True, exist_ok=True)

fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]

failed_samples = []

for file in fastq_files:
    sample_name = file.split('.fastq')[0]
    input_path = os.path.join(trimmed_dir, file)
    output_path = os.path.join(filtered_dir, f"{sample_name}_filtered.fastq")
    
    bash_cmd = (
        f"conda run -n 16s-nanopore bash -c "
        f"\"zcat -f '{input_path}' | "
        f"chopper "
        f"-q 10 "
        f"--minlength 1000 " #Is the minimum length of the reads to keep. Reads shorter than this will be discarded.
        f"--maxlength 1700 " #Is the maximum length of the reads to keep. Reads longer than this will be discarded.
        f"> '{output_path}'\""
    )
    
    print(f"Filtering {sample_name}...")
    
    try:
        # check=True Makes subprocess.run raise an exception if the command fails
        subprocess.run(bash_cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] {sample_name}!")
        print("Skipping to the next sample...\n")
        failed_samples.append(sample_name)
        
        if os.path.exists(output_path):
            os.remove(output_path)

print("\n--- Filtered end ---")
if failed_samples:
    print(f"Failed samples: {failed_samples}")
else:
    print("All samples were processed successfully!")

# Download SILVA database

In [2]:
import os
import subprocess

base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream'
silva_db_dir = os.path.join(base_out, 'silva_database')
os.makedirs(silva_db_dir, exist_ok=True)

tar_path = os.path.join(silva_db_dir, 'silva.tar')

osf_path = 'osfstorage/emu-prebuilt/silva-138.2.tar'

if not os.path.exists(tar_path):
    print("data base SILVA downloading...")
    dl_cmd = f"conda run -n 16s-nanopore osf -p 56uf7 fetch '{osf_path}' '{tar_path}'"
    subprocess.run(dl_cmd, shell=True, check=True)
    
    tar_cmd = f"tar -xvf '{tar_path}' -C '{silva_db_dir}'"
    subprocess.run(tar_cmd, shell=True, check=True)
    print("SILVA database configured successfully!")
else:
    print(f"The SILVA database already exists in {silva_db_dir}.")

The SILVA database already exists in /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/silva_database.


# Download NCBI taxonomy database

In [6]:
import os
import subprocess

base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream'
ncbi_db_dir = os.path.join(base_out, 'ncbi_database')
os.makedirs(ncbi_db_dir, exist_ok=True)
tar_path = os.path.join(ncbi_db_dir, 'emu.tar')
osf_path = 'osfstorage/emu-prebuilt/emu.tar'
check_file = os.path.join(ncbi_db_dir, 'taxonomy.tsv')
if not os.path.exists(check_file):
    print("Downloading NCBI (Emu default) database via OSF...")
    dl_cmd = f"conda run -n 16s-nanopore osf -p 56uf7 fetch '{osf_path}' '{tar_path}'"
    subprocess.run(dl_cmd, shell=True, check=True)

    print("Extracting the database...")
    tar_cmd = f"tar -xvf '{tar_path}' -C '{ncbi_db_dir}'"
    subprocess.run(tar_cmd, shell=True, check=True)
    
    print("NCBI database configured successfully!")
else:
    print(f"The NCBI database already exists in {ncbi_db_dir}.")

The NCBI database already exists in /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/ncbi_database.


# Generate a database for kma

1. Generate KMA database

In [ ]:
!mkdir -p /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_ncbi_db
!mkdir -p /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_silva_db

# ncbi
!conda run -n 16s-nanopore kma index -i /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/ncbi_database/species_taxid.fasta -o /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_ncbi_db/ncbi_db

# silva
!conda run -n 16s-nanopore kma index -i /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/silva_database/species_taxid.fasta -o /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_silva_db/silva_db

# end
!echo "Database indexing completed successfully."

2. KMA classification (SILVA)

In [ ]:
#3.kma classification
import subprocess
from pathlib import Path

input_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")
kma_out = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/silva")
kma_out.mkdir(parents=True, exist_ok=True)

# Replace with your actual KMA database path
kma_db = "/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_silva_db/silva_db"

fastq_files = [f for f in input_dir.iterdir() if f.name.endswith('.fastq')]

print(f"Starting KMA classification for {len(fastq_files)} samples...")

for file in fastq_files:
    sample_name = file.name.split('_filtered')[0]
    out_prefix = kma_out / sample_name
    
    cmd = [
        "conda", "run", "-n", "16s-nanopore",
        "kma",
        "-i", str(file),
        "-o", str(out_prefix),
        "-t_db", kma_db,
        "-bcNano",         # Optimizes for Nanopore errors
        "-ont", # Optimizes for ONT reads
        "-ID", "97.0",   # Reliability: Minimum 97% identity
        "-p", "0.01",    # Reliability: 99% statistical confidence
        "-t", "4"
    ]
    
    print(f"Running KMA for {sample_name}...")
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print(f"[ERROR] Failed to classify {sample_name}")

print("\nKMA classification finished.")

2. KMA classification (NCBI)

In [1]:
#3.kma classification
import subprocess
from pathlib import Path

input_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")
kma_out = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi")
kma_out.mkdir(parents=True, exist_ok=True)

# Replace with your actual KMA database path
kma_db = "/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_ncbi_db/ncbi_db"

fastq_files = [f for f in input_dir.iterdir() if f.name.endswith('.fastq')]

print(f"Starting KMA classification for {len(fastq_files)} samples...")

for file in fastq_files:
    sample_name = file.name.split('_filtered')[0]
    out_prefix = kma_out / sample_name
    
    cmd = [
        "conda", "run", "-n", "16s-nanopore",
        "kma",
        "-i", str(file),
        "-o", str(out_prefix),
        "-t_db", kma_db,
        "-bcNano",         # Optimizes for Nanopore errors
        "-ont",             # Specifies Oxford Nanopore reads
        "-ID", "99.0",   # Reliability: Minimum 99% identity
        "-p", "0.01",    # Reliability: 99% statistical confidence
        "-t", "4"
    ]
    
    print(f"Running KMA for {sample_name}...")
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print(f"[ERROR] Failed to classify {sample_name}")

print("\nKMA classification finished.")

Starting KMA classification for 12 samples...
Running KMA for FAT-1...


# Running KMA.
# Reading inputfile: 	/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered/FAT-1_filtered.fastq
# Phred scale:	33
#
# Total time used for DB loading: 0.09 s.
#
# Finding k-mer ankers
#
# Total number of query fragment after trimming:	2727
#
# Query converted
#
# Query ankered
#
# KMA mapping done
#
# Sort, output and select KMA alignments.
# Total time for sorting and outputting KMA alignment	0.01 s.
#
# Doing local assemblies of found templates, and output results
# Total time used for local assembly: 0.55 s.
#
# Closing files


Running KMA for DES-1...


# Running KMA.
# Reading inputfile: 	/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered/DES-1_filtered.fastq
# Phred scale:	33
#
# Total time used for DB loading: 0.06 s.
#
# Finding k-mer ankers
#
# Total number of query fragment after trimming:	2369
#
# Query converted
#
# Query ankered
#
# KMA mapping done
#
# Sort, output and select KMA alignments.
# Total time for sorting and outputting KMA alignment	0.00 s.
#
# Doing local assemblies of found templates, and output results
# Total time used for local assembly: 0.47 s.
#
# Closing files


Running KMA for FAT-3...


...

# Running KMA.
# Reading inputfile: 	/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered/MOT-2_filtered.fastq
# Phred scale:	33
#
# Total time used for DB loading: 0.06 s.
#
# Finding k-mer ankers
#
# Total number of query fragment after trimming:	4101
#
# Query converted
#
# Query ankered
#
# KMA mapping done
#
# Sort, output and select KMA alignments.
# Total time for sorting and outputting KMA alignment	0.01 s.
#
# Doing local assemblies of found templates, and output results
# Total time used for local assembly: 0.83 s.
#
# Closing files


Running KMA for CTL-3...


# Running KMA.
# Reading inputfile: 	/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered/CTL-3_filtered.fastq
# Phred scale:	33
#
# Total time used for DB loading: 0.06 s.
#
# Finding k-mer ankers
#
# Total number of query fragment after trimming:	1238
#
# Query converted
#
# Query ankered
#
# KMA mapping done
#
# Sort, output and select KMA alignments.
# Total time for sorting and outputting KMA alignment	0.00 s.
#
# Doing local assemblies of found templates, and output results
# Total time used for local assembly: 0.36 s.
#
# Closing files


Running KMA for CTL-1...

KMA classification finished.


# Running KMA.
# Reading inputfile: 	/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered/CTL-1_filtered.fastq
# Phred scale:	33
#
# Total time used for DB loading: 0.06 s.
#
# Finding k-mer ankers
#
# Total number of query fragment after trimming:	3633
#
# Query converted
#
# Query ankered
#
# KMA mapping done
#
# Sort, output and select KMA alignments.
# Total time for sorting and outputting KMA alignment	0.01 s.
#
# Doing local assemblies of found templates, and output results
# Total time used for local assembly: 0.71 s.
#
# Closing files


# Merge KMA results

In [2]:
import pandas as pd
from pathlib import Path

# 1. Define the directory where KMA saved the results
kma_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi")

# 2. Find all KMA result files (.res)
res_files = list(kma_dir.glob("*.res"))
print(f"Found {len(res_files)} KMA result files. Merging them now...")

df_list = []

# 3. Read each file and extract the taxonomy and abundance score
for file in res_files:
    sample_name = file.stem  # Gets the sample name without the .res extension
    
    # Read the tab-separated KMA output
    df = pd.read_csv(file, sep='\t')
    
    # Keep only the Taxonomy Name (#Template) and the Abundance (Score)
    df = df[['#Template', 'Score']]
    df = df.rename(columns={'Score': sample_name})
    df.set_index('#Template', inplace=True)
    
    df_list.append(df)

# 4. Combine all samples into a single matrix and fill missing values with 0
otu_matrix = pd.concat(df_list, axis=1).fillna(0)

# 5. Save the final matrix
output_file = kma_dir / "otu_abundance_matrix.csv"
otu_matrix.to_csv(output_file)

print(f"Matrix successfully saved to:\n{output_file}")

Found 12 KMA result files. Merging them now...
Matrix successfully saved to:
/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/otu_abundance_matrix.csv


1. kda classification 

In [3]:
import pandas as pd
from pathlib import Path

# 1. Define paths
matrix_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi")
raw_matrix_path = matrix_dir / "otu_abundance_matrix.csv"
clean_matrix_path = matrix_dir / "otu_abundance_matrix_CLEAN.csv"

# Path to the taxonomy mapping file inside your database folder
tax_db_path = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/ncbi_database/taxonomy.tsv")

# 2. Load the raw KMA matrix
df = pd.read_csv(raw_matrix_path)

# Extract just the tax_id from the long KMA template string
df['tax_id'] = df['#Template'].apply(lambda x: str(x).split(':')[0])
df = df.rename(columns={'#Template': 'raw_taxonomy'})

print("Loading taxonomy database dictionary...")

try:
    # 3. Load the full taxonomy database
    # Ensures tax_id is read as a string to match properly
    tax_db = pd.read_csv(tax_db_path, sep='\t', dtype={'tax_id': str})
    
    # 4. Merge (Join) the abundance matrix with the full taxonomy using the tax_id
    df_merged = pd.merge(df, tax_db, on='tax_id', how='left')
    
    # 5. Reorder columns: taxonomy first, then sample counts
    tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    
    # Find all columns that are sample names
    sample_cols = [col for col in df.columns if col not in ['tax_id', 'raw_taxonomy']]
    
    final_cols = tax_cols + sample_cols
    df_clean = df_merged[final_cols]
    
    # 6. Save the fully populated clean matrix
    df_clean.to_csv(clean_matrix_path, index=False)
    
    print("Data cleaning and taxonomy mapping finished successfully.")
    print(f"Clean matrix saved to: {clean_matrix_path}")

except FileNotFoundError:
    print(f"ERROR: Could not find the taxonomy.tsv file at {tax_db_path}")
    print("Please make sure the path to your database taxonomy file is correct.")

Loading taxonomy database dictionary...
Data cleaning and taxonomy mapping finished successfully.
Clean matrix saved to: /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/otu_abundance_matrix_CLEAN.csv


## Creating different tables for analysis

In [5]:
import pandas as pd
import skbio
import os


# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/otu_abundance_matrix_CLEAN.csv'
meta_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_16s.csv'

# 2. Load OTU matrix and isolate taxonomy
otu_df = pd.read_csv(matrix_path)
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']

# Save taxonomy hierarchy separately
taxonomy_df = otu_df[tax_cols].copy().set_index('species')

# 3. Prepare Raw Counts (Genus level)
counts_df = otu_df.drop(columns=[c for c in tax_cols if c != 'genus'])
counts_df = counts_df.groupby('genus').sum(numeric_only=True).T.astype(int)

# 4. Load Metadata and align samples
meta_df = pd.read_csv(meta_path).set_index('sample_title')
common_samples = counts_df.index.intersection(meta_df.index)
counts_df = counts_df.loc[common_samples]
meta_df = meta_df.loc[common_samples]

print("Setup complete!")
print(f"Samples ready: {counts_df.shape[0]}")
print(f"Unique Genera: {counts_df.shape[1]}\n")

# --- NOVO: Verificação de Profundidade ---
sample_totals = counts_df.sum(axis=1)
print("Total reads per sample:")
print(sample_totals.sort_values())

# 5. Rarefy counts for Alpha Diversity
rarefaction_depth = 344700

# Filtered samples based on rarefaction depth 
valid_samples = sample_totals[sample_totals >= rarefaction_depth].index
dropped_samples = set(counts_df.index) - set(valid_samples)

if dropped_samples:
    print(f"\n The samples with less than {rarefaction_depth} reads were removed: {dropped_samples}")

counts_df_valid = counts_df.loc[valid_samples]

counts_rarefied = counts_df_valid.apply(
    lambda x: skbio.stats.subsample_counts(x.values, rarefaction_depth), 
    axis=1, 
    result_type='broadcast'
)
counts_rarefied.columns = counts_df_valid.columns

print(f"\nRarefied matrix created (Depth: {rarefaction_depth} reads/sample).")

Setup complete!
Samples ready: 12
Unique Genera: 6

Total reads per sample:
CTL-2     344758
DES-2     477565
MOT-1     675101
CTL-3     933738
FAT-3    1264026
FAT-2    1286955
FAT-1    1510857
DES-1    1664400
MOT-3    1819601
CTL-1    2168672
MOT-2    2256304
DES-3    3935414
dtype: int64

Rarefied matrix created (Depth: 344700 reads/sample).


In [6]:
import os

# 1. Define output directory
out_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/5_statistics/kda'
os.makedirs(out_dir, exist_ok=True)

# 2. Filter metadata to match valid rarefied samples
meta_df_valid = meta_df.loc[valid_samples]

# 3. Save filtered metadata
meta_out_path = os.path.join(out_dir, 'metadata_filtered.csv')
meta_df_valid.to_csv(meta_out_path)

# 4. Save rarefied genus abundance counts
counts_out_path = os.path.join(out_dir, 'rarefied_genus_counts.csv')
counts_rarefied.to_csv(counts_out_path)

print(f"Tables successfully saved to:\n{out_dir}")

Tables successfully saved to:
/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/5_statistics/kda


## Amcombc.r

In [ ]:
!conda create -n env_ancombc -c conda-forge -c bioconda bioconductor-ancombc bioconductor-phyloseq bioconductor-microbiome r-tidyverse -y
!conda activate env_ancombc